# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rset in record_sets:
    print(f"Record set @id: {rset.id}")
    print(f"  Name: {rset.name}")
    print(f"  Description: {rset.description}")
    print(f"  Fields:")
    for field in rset.fields:
        print(f"    - {field.id}: {field.name} ({field.data_type})")
    print("---\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all dataframes for each record set
dataframes = {}
record_set_ids = [rset.id for rset in metadata.record_sets]

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

# Display columns of the first record set
if record_set_ids:
    first_set_id = record_set_ids[0]
    print(f"Columns for record set {first_set_id}:")
    print(dataframes[first_set_id].columns.tolist())
    dataframes[first_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data, or grouping by attributes.

In [ ]:
# Example EDA for numeric fields
import numpy as np

if record_set_ids:
    df = dataframes[first_set_id].copy()

    # Try to find a numeric field
    numeric_fields = []
    for field in [rf for rf in metadata.record_sets[0].fields]:
        if field.data_type in ('http://schema.org/Float', 'http://schema.org/Integer', 'Float', 'Integer', 'Number'):
            numeric_fields.append(field.id)

    print(f"Numeric fields: {numeric_fields}")
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]
        # Cleaning column names to match field IDs in dataframe
        col_map = {col: col for col in df.columns}
        if numeric_field_id not in df.columns:
            # Try to find the correct column
            for col in df.columns:
                if numeric_field_id in col or col in numeric_field_id:
                    col_map[numeric_field_id] = col
                    break
        colname = col_map.get(numeric_field_id, numeric_field_id)
        # Convert to numeric if necessary
        df[colname] = pd.to_numeric(df[colname], errors='coerce')
        # Filter for values above a threshold, e.g. 10
        threshold = 10
        filtered_df = df[df[colname] > threshold]
        print(f"Filtered records with {colname} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{colname}_normalized"] = (filtered_df[colname] - filtered_df[colname].mean()) / filtered_df[colname].std()
        print(f"Normalized {colname} for filtered records:")
        print(filtered_df[[colname, f"{colname}_normalized"]].head())

        # Try grouping by a categorical field if present
        group_field_id = None
        for field in metadata.record_sets[0].fields:
            if field.data_type in ('http://schema.org/Text', 'Text', 'http://schema.org/DefinedTerm', 'DefinedTerm'):
                if field.id != numeric_field_id:
                    group_field_id = field.id
                    break
        if group_field_id:
            group_colname = col_map.get(group_field_id, group_field_id)
            if group_colname in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_colname)[colname].mean().reset_index().sort_values(by=colname, ascending=False)
                print(f"Grouped data by {group_colname} (mean of {colname}):")
                print(grouped_df.head())
    else:
        print("No numeric fields found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize distribution of first numeric field (if present)
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and len(numeric_fields) > 0:
    colname = col_map.get(numeric_fields[0], numeric_fields[0])
    plt.figure(figsize=(8,4))
    sns.histplot(df[colname].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {colname}")
    plt.xlabel(colname)
    plt.show()

    # If grouping field exists, boxplot
    group_field_name = group_field_id if group_field_id else None
    if group_field_name and group_field_name in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field_name, y=colname, data=df)
        plt.title(f"{colname} by {group_field_name}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the FAIR^2 mass spectrometry dataset using `mlcroissant`. Further analysis tailored to the experiment's hypothesis can build on these techniques.